# Smartphone Addiction - GPU Stacking Pipeline

XGBoost + CatBoost + LightGBM + RealMLP, 5-fold OOF stacking with a logistic-regression
meta-learner. Optuna tunes the three GBDTs; RealMLP uses its tuned defaults. 2-seed bagging,
IterativeImputer fitted inside each fold, whole run bounded by `MAX_HOURS`.

Setup guide: `KAGGLE_SETUP.md`.

In [ ]:
# Install only. Must run before numpy/torch/xgboost are imported: pip-upgrading an
# already-imported package leaves the kernel with mismatched binaries, and a committed
# Kaggle run cannot restart its kernel.
import os, sys, subprocess, importlib

USE_MLP = True

def _install_mlp():
    if not USE_MLP:
        return "disabled by USE_MLP"
    try:
        importlib.import_module("pytabkit")
        return "already present"
    except ImportError:
        pass
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir",
                        "pytabkit==1.7.3"], capture_output=True, text=True, timeout=1800)
    importlib.invalidate_caches()
    if r.returncode != 0:
        return f"pip failed: {r.stderr.strip()[-300:]}"
    try:
        importlib.import_module("pytabkit")
        return "installed pytabkit==1.7.3"
    except Exception as e:
        return f"installed but import failed: {type(e).__name__}: {e}"

print("pytabkit:", _install_mlp())

In [ ]:
import gc, glob, logging, subprocess, time, warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy.special import logit
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

HAS_MLP = False
if USE_MLP:
    try:
        from pytabkit import RealMLP_TD_Classifier
        HAS_MLP = True
    except Exception as e:
        print(f"RealMLP unavailable -> 3-model stack. {type(e).__name__}: {e}")

# pytorch-lightning logs 3 lines per fit regardless of verbosity=0
for _n in [n for n in logging.root.manager.loggerDict if "lightning" in n.lower()]:
    _lg = logging.getLogger(_n); _lg.setLevel(logging.CRITICAL); _lg.propagate = False

SEED = 42
SEEDS = [42, 2024]
N_SPLITS = 5
ESR = 300
IMP_ITER = 20

MAX_HOURS = 8.0
TUNE_HOURS = 3.0
TUNE_SAMPLE = 200_000
SEL_SAMPLE = 200_000
TUNE_FOLDS = 3
TUNE_PATIENCE = 15
MLP_EPOCHS = 40

TARGET = "addicted_label"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]
NUM_COLS = ["age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
            "work_study_hours", "sleep_hours", "notifications_per_day",
            "app_opens_per_day", "weekend_screen_time"]

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
np.random.seed(SEED)

T0 = time.time()
def elapsed_s():   return time.time() - T0
def remaining_s(): return MAX_HOURS * 3600 - elapsed_s()

print(f"RealMLP available: {HAS_MLP}   budget {MAX_HOURS}h (tune {TUNE_HOURS}h)")

In [ ]:
def has_cuda():
    try:
        subprocess.check_output(["nvidia-smi"], stderr=subprocess.DEVNULL)
        return True
    except Exception:
        return False

def lgb_gpu_works():
    if not has_cuda():
        return False
    try:
        lgb.LGBMClassifier(n_estimators=2, device="gpu", verbose=-1).fit(
            np.random.rand(64, 4), np.random.randint(0, 2, 64))
        return True
    except Exception:
        return False

def free_gpu():
    # torch's caching allocator keeps ~2GB after a RealMLP fit; CatBoost-GPU at depth 8
    # runs next and is the documented OOM candidate, so hand the memory back.
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

CUDA = has_cuda()
GPU = {"xgb": CUDA, "cat": CUDA, "lgb": lgb_gpu_works()}
N_JOBS = os.cpu_count()
print(f"cuda={CUDA}  gpu={GPU}  cores={N_JOBS}")

In [ ]:
def find_data():
    known = ["/kaggle/input/competitions/playground-series-s6e8",
             "/kaggle/input/playground-series-s6e8", "../data", "data"]
    cands = [f"{d}/train.csv" for d in known]
    cands += sorted(glob.glob("/kaggle/input/**/train.csv", recursive=True))
    for f in cands:
        d = os.path.dirname(f)
        if os.path.exists(f) and os.path.exists(f"{d}/test.csv"):
            return d
    seen = sorted(glob.glob("/kaggle/input/**", recursive=True))[:40]
    raise FileNotFoundError("train.csv + test.csv not found. /kaggle/input holds:\n" + "\n".join(seen))

DATA = find_data()
train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")

y = train.pop(TARGET).values
test_ids = test["id"].values
X = train.drop(columns="id")
X_test = test.drop(columns="id")[X.columns]

for d in (X, X_test):
    for c in NUM_COLS:
        d[c] = d[c].astype("float32")

for c in CAT_COLS:
    levels = sorted(set(X[c].dropna()) | set(X_test[c].dropna()))
    X[c] = pd.Categorical(X[c], categories=levels)
    X_test[c] = pd.Categorical(X_test[c], categories=levels)

print(f"train {X.shape}  test {X_test.shape}  base rate {y.mean():.4f}  nan {X.isna().sum().sum():,}")

In [ ]:
IMP_COLS = [c + "_imp" for c in NUM_COLS]
IND_COLS = [c + "_na" for c in NUM_COLS]
MLP_CAT_COLS = CAT_COLS + IND_COLS
MLP_COLS = IMP_COLS + MLP_CAT_COLS

# Pinned once from the train+test feature union so every fold's train/valid/test frames
# encode to identical integer codes. Independent .astype("category") calls would shift
# the codes between frames and silently corrupt the net's input.
CAT_LEVELS = {c: [str(v) for v in X[c].cat.categories] + ["missing"] for c in CAT_COLS}
IND_LEVELS = ["0", "1"]

def add_imputed(X_fit, frames, verbose=True):
    imp = IterativeImputer(max_iter=IMP_ITER, random_state=SEED).fit(X_fit[NUM_COLS])
    if verbose:
        print(f"  imputer n_iter={imp.n_iter_}/{IMP_ITER}"
              + ("  <-- did not converge" if imp.n_iter_ >= IMP_ITER else ""))
    out = []
    for d in frames:
        d = d.copy()
        d[IMP_COLS] = imp.transform(d[NUM_COLS]).astype("float32")
        out.append(d)
    return out

def to_catboost(d):
    d = d.copy()
    for c in CAT_COLS:
        d[c] = d[c].astype(object).fillna("missing").astype(str)
    return d

def to_mlp(d):
    # RealMLP cannot represent NaN, so it gets the imputed values plus explicit
    # missingness indicators. Indicators are categorical, not float: RealMLP robust-scales
    # numerics by median/IQR and a binary column with ~4% ones has IQR=0.
    out = pd.DataFrame(index=d.index)
    for ic in IMP_COLS:
        out[ic] = d[ic].astype("float32")
    for c in CAT_COLS:
        v = d[c].astype(object).fillna("missing").astype(str)
        out[c] = pd.Categorical(v, categories=CAT_LEVELS[c])
    for c, nc in zip(NUM_COLS, IND_COLS):
        out[nc] = pd.Categorical(np.where(d[c].isna(), "1", "0"), categories=IND_LEVELS)
    out = out[MLP_COLS]
    assert not out[IMP_COLS].isna().any().any(), "NaN reached the MLP numeric block"
    assert not out[MLP_CAT_COLS].isna().any().any(), "unseen category level in MLP frame"
    return out

def L(p):
    return logit(np.clip(p, 1e-6, 1 - 1e-6))

In [ ]:
DEFAULTS = {
    "xgb": dict(max_depth=8, min_child_weight=20, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=2.0),
    "cat": dict(depth=8, l2_leaf_reg=6.0, random_strength=1.0,
                bagging_temperature=1.0, min_data_in_leaf=1),
    "lgb": dict(num_leaves=96, min_child_samples=50, subsample=0.8,
                colsample_bytree=0.8, reg_lambda=2.0),
}

def make_xgb(seed, p=None):
    return xgb.XGBClassifier(
        n_estimators=30000, learning_rate=0.02, eval_metric="auc",
        early_stopping_rounds=ESR, enable_categorical=True, tree_method="hist",
        device="cuda" if GPU["xgb"] else "cpu", n_jobs=N_JOBS, random_state=seed,
        **{**DEFAULTS["xgb"], **(p or {})})

def make_cat(seed, p=None):
    return cb.CatBoostClassifier(
        iterations=30000, learning_rate=0.03, eval_metric="AUC", random_seed=seed,
        verbose=False, allow_writing_files=False,
        task_type="GPU" if GPU["cat"] else "CPU",
        **{**DEFAULTS["cat"], **(p or {})})

def make_lgb(seed, p=None):
    q = dict(n_estimators=20000, learning_rate=0.03, objective="binary",
             subsample_freq=1, random_state=seed, n_jobs=N_JOBS, verbose=-1)
    q.update({**DEFAULTS["lgb"], **(p or {})})
    if GPU["lgb"]:
        q.update(device="gpu", max_bin=255)
    return lgb.LGBMClassifier(**q)

def make_mlp(seed, p=None, n_epochs=None):
    # batch_size and lr deliberately left at the TD defaults: batch size sets the
    # gradient-noise scale that the tuned lr is calibrated against. n_epochs is the
    # safe knob -- RealMLP's schedule is a function of training fraction, so a shorter
    # run compresses the anneal rather than truncating it.
    return RealMLP_TD_Classifier(
        device="cuda" if CUDA else "cpu", random_state=seed, n_cv=1,
        n_epochs=n_epochs or MLP_EPOCHS, val_metric_name="1-auc_ovr",
        n_threads=N_JOBS, verbosity=0)


def fit_xgb(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m

def fit_cat(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=(Xva, yva), cat_features=CAT_COLS,
          early_stopping_rounds=ESR, use_best_model=True, verbose=False)
    return m

def fit_lgb(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
          callbacks=[lgb.early_stopping(ESR, verbose=False)])
    return m

def fit_mlp(m, Xtr, ytr, Xva, yva):
    m.fit(Xtr, np.asarray(ytr).astype("int64"), Xva, np.asarray(yva).astype("int64"),
          cat_col_names=MLP_CAT_COLS)
    assert list(getattr(m, "classes_", [0, 1])) == [0, 1], "unexpected class order"
    return m


def best_iter(m):
    for a in ("best_iteration", "best_iteration_"):
        v = getattr(m, a, None)
        if isinstance(v, (int, np.integer)):
            return int(v)
    try:
        return int(m.get_best_iteration())
    except Exception:
        return None

def tree_cap(m):
    p = m.get_params()
    return p.get("n_estimators") or p.get("iterations")


def mlp_smoke_test(n=512):
    # Prove fit+predict_proba work on this exact dtype/level layout in ~10s rather than
    # discovering an API mismatch 30 minutes into fold 1. Synthetic data: no leakage surface.
    try:
        rng = np.random.default_rng(SEED)
        d = pd.DataFrame({c: rng.normal(size=n).astype("float32") for c in IMP_COLS})
        for c in CAT_COLS:
            d[c] = pd.Categorical(rng.choice(CAT_LEVELS[c], n), categories=CAT_LEVELS[c])
        for c in IND_COLS:
            d[c] = pd.Categorical(rng.choice(IND_LEVELS, n), categories=IND_LEVELS)
        d = d[MLP_COLS]
        t = (rng.random(n) < 0.7).astype("int64")
        t[:2], t[-2:] = [0, 1], [0, 1]
        m = fit_mlp(make_mlp(SEED, n_epochs=2), d.iloc[:400], t[:400], d.iloc[400:], t[400:])
        p = m.predict_proba(d.iloc[400:])[:, 1]
        ok = p.shape == (n - 400,) and np.isfinite(p).all()
        del m; gc.collect(); free_gpu()
        return bool(ok)
    except Exception as e:
        print(f"RealMLP smoke test FAILED -> dropping the model. {type(e).__name__}: {e}")
        return False


MODELS = {
    "xgb": (make_xgb, fit_xgb, "num"),
    "cat": (make_cat, fit_cat, "cat"),
    "lgb": (make_lgb, fit_lgb, "num"),
}
if HAS_MLP and mlp_smoke_test():
    MODELS["mlp"] = (make_mlp, fit_mlp, "mlp")

NAMES = list(MODELS)
TUNE_MODELS = [n for n in ("xgb", "cat", "lgb") if n in MODELS]
print("models in this run:", NAMES)

In [ ]:
SPACES = {
    "xgb": lambda t: dict(
        max_depth        = t.suggest_int("max_depth", 4, 12),
        min_child_weight = t.suggest_float("min_child_weight", 1, 100, log=True),
        subsample        = t.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree = t.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_lambda       = t.suggest_float("reg_lambda", 0.1, 50, log=True)),
    "cat": lambda t: dict(
        depth               = t.suggest_int("depth", 4, 10),
        l2_leaf_reg         = t.suggest_float("l2_leaf_reg", 1, 20, log=True),
        random_strength     = t.suggest_float("random_strength", 0, 5),
        bagging_temperature = t.suggest_float("bagging_temperature", 0, 2),
        min_data_in_leaf    = t.suggest_int("min_data_in_leaf", 1, 100, log=True)),
    "lgb": lambda t: dict(
        num_leaves        = t.suggest_int("num_leaves", 31, 255, log=True),
        min_child_samples = t.suggest_int("min_child_samples", 10, 200, log=True),
        subsample         = t.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree  = t.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_lambda        = t.suggest_float("reg_lambda", 0.1, 50, log=True)),
}

def build_folds(rows, n_splits):
    Xr = X.iloc[rows].reset_index(drop=True)
    yr = y[rows]
    out = []
    for tr, va in StratifiedKFold(n_splits, shuffle=True, random_state=SEED).split(Xr, yr):
        A, B = add_imputed(Xr.iloc[tr], [Xr.iloc[tr], Xr.iloc[va]], verbose=False)
        out.append({"num": (A, B), "cat": tuple(to_catboost(d) for d in (A, B)),
                    "y": (yr[tr], yr[va])})
    return out

def score_params(name, p, folds):
    make, fit, kind = MODELS[name]
    aucs = []
    for f in folds:
        A, B = f[kind]
        ytr, yva = f["y"]
        m = fit(make(SEED, p), A, ytr, B, yva)
        aucs.append(roc_auc_score(yva, m.predict_proba(B)[:, 1]))
        del m; gc.collect(); free_gpu()
    return float(np.mean(aucs))

def make_stopper(patience):
    st = {"best": -np.inf, "since": 0}
    def cb(study, trial):
        try:
            v = study.best_value
        except ValueError:
            return
        if v > st["best"] + 1e-6:
            st["best"], st["since"] = v, 0
        else:
            st["since"] += 1
            if st["since"] >= patience:
                study.stop()
    return cb

FINAL_PARAMS = {n: dict(DEFAULTS[n]) for n in DEFAULTS}

if TUNE_HOURS > 0 and TUNE_MODELS:
    perm = np.random.RandomState(SEED).permutation(len(X))
    rows_A = perm[:TUNE_SAMPLE]
    rows_B = perm[TUNE_SAMPLE:TUNE_SAMPLE + SEL_SAMPLE]
    print(f"building tuning folds (A={len(rows_A):,}) and selection folds (B={len(rows_B):,})")
    FOLDS_A = build_folds(rows_A, TUNE_FOLDS)
    FOLDS_B = build_folds(rows_B, TUNE_FOLDS)

    budget = TUNE_HOURS * 3600 / len(TUNE_MODELS)
    for name in TUNE_MODELS:
        if remaining_s() < budget * 0.5:
            print(f"{name}: out of budget, keeping defaults")
            continue
        t_m = time.time()
        study = optuna.create_study(
            direction="maximize",
            sampler=optuna.samplers.TPESampler(seed=SEED, multivariate=True))
        study.optimize(lambda tr, n=name: score_params(n, SPACES[n](tr), FOLDS_A),
                       timeout=min(budget, remaining_s() * 0.5),
                       callbacks=[make_stopper(TUNE_PATIENCE)],
                       catch=(Exception,))

        done = [t for t in study.trials if t.value is not None]
        if not done:
            print(f"{name}: no trial completed, keeping defaults")
            continue

        # Best-of-k selection bias: with k trials and per-trial noise s, the winner's
        # score is inflated by ~s*sqrt(2 ln k). Re-score the top 3 AND the defaults on
        # disjoint rows B, and only adopt tuned params if they actually win there.
        top = sorted(done, key=lambda t: t.value, reverse=True)[:3]
        cands = [("defaults", dict(DEFAULTS[name]))]
        cands += [(f"trial{t.number}", {**DEFAULTS[name], **t.params}) for t in top]
        scored = sorted(((score_params(name, p, FOLDS_B), lbl, p) for lbl, p in cands),
                        key=lambda z: z[0], reverse=True)

        print(f"\n{name}: {len(done)} trials in {(time.time()-t_m)/60:.1f} min, "
              f"study best (on A) {study.best_value:.5f}")
        for auc, lbl, _ in scored:
            print(f"   re-eval on B  {lbl:10s} {auc:.5f}")
        FINAL_PARAMS[name] = scored[0][2]
        print(f"   -> adopted: {scored[0][1]}")

    del FOLDS_A, FOLDS_B
    gc.collect(); free_gpu()

print("\nfinal params:")
for n in TUNE_MODELS:
    print(f"  {n}: {FINAL_PARAMS[n]}")
print(f"elapsed {elapsed_s()/60:.1f} min")

In [ ]:
state = {"oof": np.zeros((len(X), len(NAMES))), "mask": np.zeros(len(X), dtype=bool),
         "test_sum": np.zeros((len(X_test), len(NAMES))), "done": [], "dead": [],
         "scores": {n: [] for n in NAMES}}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
t_loop = time.time()
ran = 0

for fold, (tr, va) in enumerate(skf.split(X, y)):
    if ran:
        avg = (time.time() - t_loop) / ran
        if elapsed_s() + avg > MAX_HOURS * 3600:
            print(f"next fold would exceed {MAX_HOURS}h - stopping with {len(state['done'])} folds")
            break

    t0 = time.time()
    ytr, yva = y[tr], y[va]
    Xtr, Xva, Xte = add_imputed(X.iloc[tr], [X.iloc[tr], X.iloc[va], X_test])
    kinds = {k for n, (_, _, k) in MODELS.items() if n not in state["dead"]}
    frames = {"num": (Xtr, Xva, Xte)}
    if "cat" in kinds: frames["cat"] = tuple(to_catboost(d) for d in (Xtr, Xva, Xte))
    if "mlp" in kinds: frames["mlp"] = tuple(to_mlp(d) for d in (Xtr, Xva, Xte))

    for j, name in enumerate(NAMES):
        if name in state["dead"]:
            continue
        make, fit, kind = MODELS[name]
        A, B, C = frames[kind]
        try:
            pv, pt = np.zeros(len(va)), np.zeros(len(X_test))
            bis, cap = [], None
            for s in SEEDS:
                m = fit(make(s, FINAL_PARAMS.get(name)), A, ytr, B, yva)
                pv += m.predict_proba(B)[:, 1] / len(SEEDS)
                pt += m.predict_proba(C)[:, 1] / len(SEEDS)
                bis.append(best_iter(m)); cap = tree_cap(m)
                del m; gc.collect(); free_gpu()
        except Exception as e:
            # A half-filled OOF column would silently poison the meta-learner, so a model
            # that fails on any fold is dropped from the stack entirely.
            state["dead"].append(name); free_gpu()
            print(f"  fold {fold+1} {name:4s} FAILED -> dropped. {type(e).__name__}: {e}")
            continue

        state["oof"][va, j] = pv
        state["test_sum"][:, j] += pt
        state["scores"][name].append(roc_auc_score(yva, pv))
        hit = cap and any(b is not None and b >= cap - 1 for b in bis)
        flag = f"  <-- HIT CAP {cap}, NOT CONVERGED" if hit else ""
        print(f"  fold {fold + 1} {name:4s} AUC={state['scores'][name][-1]:.5f}  "
              f"best_iter={bis}  ({time.time() - t0:.0f}s){flag}")

    state["mask"][va] = True
    state["done"].append(fold)
    ran += 1
    del Xtr, Xva, Xte, frames
    gc.collect(); free_gpu()
    print(f"fold {fold + 1}/{N_SPLITS} done in {time.time() - t0:.0f}s")

print(f"\ntotal {elapsed_s()/60:.1f} min, {len(state['done'])} folds")

In [ ]:
assert state["done"], "no folds completed"
dead = set(state["dead"])
LIVE = [n for n in NAMES if n not in dead]
COLS = [NAMES.index(n) for n in LIVE]
assert LIVE, "every model failed"
if dead:
    print(f"dropped during training: {sorted(dead)}")

mask = state["mask"]
oof = state["oof"][mask][:, COLS]
y_oof = y[mask]
test_p = (state["test_sum"] / len(state["done"]))[:, COLS]

for j, n in enumerate(LIVE):
    print(f"{n:4s} OOF AUC {roc_auc_score(y_oof, oof[:, j]):.5f}   "
          f"folds {np.round(state['scores'][n], 5)}")
print(f"mean OOF AUC {roc_auc_score(y_oof, oof.mean(1)):.5f}")

# A stack gains from decorrelation, not from the new model being individually strong.
# The GBDTs should sit ~0.98+ with each other; if RealMLP is also >0.98 it adds nothing.
print("\nOOF logit correlation:")
print(pd.DataFrame(np.corrcoef(L(oof).T), index=LIVE, columns=LIVE).round(4).to_string())

In [ ]:
meta = LogisticRegression(C=1.0, max_iter=1000)
honest = cross_val_predict(meta, L(oof), y_oof, cv=5, method="predict_proba")[:, 1]
print(f"stack OOF AUC {roc_auc_score(y_oof, honest):.5f}  (meta cross-validated)")

meta.fit(L(oof), y_oof)
print("meta weights:", dict(zip(LIVE, meta.coef_[0].round(3))))

In [ ]:
pred = meta.predict_proba(L(test_p))[:, 1]

sub = pd.DataFrame({"id": test_ids, TARGET: pred})
assert len(sub) == len(X_test), f"expected {len(X_test)} rows, got {len(sub)}"
assert sub["id"].is_unique
assert sub[TARGET].notna().all()
assert sub[TARGET].between(0, 1).all()
assert sub[TARGET].nunique() > 2

sub.to_csv(f"{WORK}/submission.csv", index=False)
np.savez(f"{WORK}/oof_stack.npz", oof=state["oof"], mask=state["mask"],
         test=state["test_sum"] / len(state["done"]),
         names=np.array(NAMES), live=np.array(LIVE), dead=np.array(sorted(dead)))

print(f"{len(sub):,} rows  mean {pred.mean():.4f}  (train base rate {y.mean():.4f})")
if abs(pred.mean() - y.mean()) > 0.03:
    print("WARNING: mean prediction far from base rate - inspect before submitting")
print(f"folds used: {len(state['done'])}/{N_SPLITS}   models used: {LIVE}")
sub.head()